<a href="https://colab.research.google.com/github/hussainaarish7/Development-of-a-Conversational-AI-for-IT-Support-Using-RAG-and-Fine-Tuned-Language-Models/blob/main/Development_of_a_Conversational_AI_for_IT_Support_Using_RAG_and_Fine_Tuned_Language_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IT Support Chatbot with RAG

**Project Created By**-  Mohd Aarish

# **Project Summary -**

The project focuses on the development of an intelligent Conversational AI system for IT support using advanced Natural Language Processing techniques. The primary goal is to automate technical support services by enabling a chatbot to understand user queries and provide accurate, context-aware solutions in real time.

The system is built using a Retrieval-Augmented Generation (RAG) framework, which combines the strengths of information retrieval and generative language models. A domain-specific dataset consisting of IT support queries, troubleshooting guides, and FAQs is used to train the system. The data is preprocessed, cleaned, and converted into vector embeddings using transformer-based models. These embeddings are stored in a vector database (FAISS), allowing efficient similarity-based retrieval of relevant information.

When a user submits a query, the system retrieves the most relevant context from the knowledge base and feeds it into a pre-trained language model to generate a meaningful and accurate response. This approach significantly improves response quality compared to traditional rule-based chatbots.

Exploratory Data Analysis (EDA) is performed on the dataset to understand query patterns, text distribution, and common issues faced by users. The system is evaluated based on metrics such as response accuracy, relevance, and response time.

The developed chatbot demonstrates strong performance in handling common IT issues such as system slowdowns, connectivity problems, and software errors. It reduces dependency on human support agents, improves response time, and enhances user experience.

Overall, this project highlights the practical application of AI in automating IT support services and provides a scalable solution that can be integrated into real-world enterprise systems.

# **GitHub Link -**
https://github.com/hussainaarish7/Development-of-a-Conversational-AI-for-IT-Support-Using-RAG-and-Fine-Tuned-Language-Models

# **Problem Statement**


Modern organizations rely heavily on IT infrastructure, making efficient and timely technical support essential for maintaining productivity and operational continuity. However, existing IT support systems are largely dependent on human agents and traditional rule-based chatbots, which suffer from several limitations. These systems often struggle to handle a high volume of repetitive queries, leading to increased response time, higher operational costs, and inconsistent user experience.

Rule-based chatbots, in particular, lack the ability to understand natural language context and fail when dealing with complex, ambiguous, or multi-step technical queries. They operate on predefined rules and keyword matching, which limits their scalability and adaptability in dynamic IT environments. As a result, users frequently experience frustration due to inaccurate or incomplete responses, ultimately increasing reliance on human support teams.

Furthermore, existing solutions do not effectively utilize large volumes of unstructured IT support data such as documentation, troubleshooting guides, and historical tickets. This leads to underutilization of valuable knowledge resources that could otherwise improve support quality and efficiency.

Therefore, the key problem addressed in this project is the development of an intelligent, scalable, and context-aware conversational AI system that can accurately understand user queries, retrieve relevant information from domain-specific knowledge bases, and generate meaningful responses. The system aims to overcome the limitations of traditional approaches by leveraging Retrieval-Augmented Generation (RAG) and advanced language models to enhance response accuracy, reduce resolution time, and improve overall user satisfaction in IT support services.

# ***Let's Begin !***

## 1. Setup and Library Installation

First, we install the necessary libraries for data manipulation, embedding generation, similarity search, and text generation.

In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn
!pip install -q transformers sentence-transformers faiss-cpu
!pip install -q langchain gradio datasets accelerate


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

import faiss

import torch
print("GPU Available:", torch.cuda.is_available())


This cell imports all the necessary Python libraries. Let's break them down:

*   **`pandas`** and **`numpy`**: These are fundamental libraries for data manipulation and numerical operations, crucial for handling our dataset.
*   **`matplotlib.pyplot`** and **`seaborn`**: Used for data visualization, helping us understand the data distribution through plots and charts.
*   **`datasets`** (from Hugging Face): A library to easily load and work with various datasets, though here we're using a custom dictionary to create a DataFrame.
*   **`sentence_transformers`**: Essential for converting text (our queries) into numerical vector embeddings, which capture the semantic meaning of the text.
*   **`transformers`**: Provides access to pre-trained models like GPT-2 for text generation and `pipeline` for easily using these models.
*   **`faiss`**: A library for efficient similarity search among dense vectors, forming the core of our Retrieval-Augmented Generation (RAG) system.
*   **`torch`**: The underlying library for deep learning operations, used here to check for GPU availability, which can speed up model computations.

Libraries Used and Their Purpose
pandas, numpy → Data manipulation and numerical operations
matplotlib, seaborn → Data visualization and EDA
scikit-learn → Preprocessing and evaluation metrics
transformers → Pre-trained language models (LLMs)
sentence-transformers → Text embeddings generation
faiss → Efficient vector similarity search (core of RAG)
langchain → Framework for building conversational pipelines
gradio → Web-based user interface for chatbot interaction
datasets → Loading real-world datasets

In [ ]:
data = {
    "query": [
        # 🔹 System Performance
        "System is running slow",
        "Computer is freezing frequently",
        "PC taking too long to boot",
        "Applications are lagging",

        # 🔹 Internet & Network
        "Internet is not working",
        "WiFi keeps disconnecting",
        "Unable to connect to WiFi",
        "Slow internet speed",
        "Network connection lost",

        # 🔹 Hardware Issues
        "Laptop overheating",
        "Fan is making noise",
        "Keyboard not responding",
        "Mouse not working",
        "Battery draining quickly",

        # 🔹 Display Issues
        "Screen flickering issue",
        "Display not turning on",
        "Low screen brightness",
        "External monitor not detected",

        # 🔹 Software Issues
        "Software crashing frequently",
        "Application not opening",
        "Error while installing software",
        "Program not responding",

        # 🔹 Storage Issues
        "Low disk space",
        "Unable to save files",
        "Hard drive not detected",

        # 🔹 Security Issues
        "Virus detected on system",
        "System infected with malware",
        "Antivirus not updating",

        # 🔹 Login Issues
        "Forgot system password",
        "Unable to login",
        "Account locked",

        # 🔹 Printer Issues
        "Printer not working",
        "Printer not connecting",
        "Print jobs stuck in queue",

        # 🔹 Audio Issues
        "No sound from speakers",
        "Microphone not working",

        # 🔹 Updates
        "Windows update failed",
        "System not updating",

        # 🔹 General
        "System shutting down automatically",
        "Blue screen error",
        "USB device not recognized"
    ],

    "solution": [
        # System Performance
        "Check background apps and restart system",
        "Close unnecessary programs and scan for malware",
        "Disable startup programs and upgrade RAM if needed",
        "Update software and check CPU usage",

        # Internet
        "Restart router and check cables",
        "Move closer to router and reset network settings",
        "Reconnect network or reset WiFi settings",
        "Check bandwidth usage and contact ISP",
        "Restart network adapter",

        # Hardware
        "Clean fans and improve ventilation",
        "Clean or replace the fan",
        "Reconnect keyboard or update drivers",
        "Check mouse connection or replace battery",
        "Reduce background apps and check battery health",

        # Display
        "Update display drivers",
        "Check power cable and display settings",
        "Adjust brightness settings",
        "Reconnect monitor and update drivers",

        # Software
        "Reinstall or update software",
        "Check compatibility and reinstall",
        "Run installer as administrator",
        "Force close and restart application",

        # Storage
        "Delete unnecessary files and clear cache",
        "Check permissions and disk health",
        "Check cable connection or replace drive",

        # Security
        "Run full antivirus scan",
        "Install trusted antivirus and remove malware",
        "Update antivirus software",

        # Login
        "Reset password using recovery option",
        "Check credentials and try again",
        "Contact admin to unlock account",

        # Printer
        "Check printer connection and drivers",
        "Reconnect printer to network",
        "Clear print queue and restart printer",

        # Audio
        "Check volume and audio drivers",
        "Check microphone permissions",

        # Updates
        "Restart system and retry update",
        "Check internet and update settings",

        # General
        "Check power settings and overheating",
        "Check error code and update drivers",
        "Reconnect USB device or update drivers"
    ]
}

df = pd.DataFrame(data)
df.head()


This code block defines our small, custom dataset of IT support queries and their corresponding solutions. It's structured as a dictionary with two keys: `query` and `solution`. Each list contains various IT-related problems and their direct fixes.

*   **`query`**: Contains common issues like 'System is running slow', 'Internet is not working', 'Laptop overheating', etc.
*   **`solution`**: Provides a concise solution for each corresponding query.

Finally, `pd.DataFrame(data)` converts this dictionary into a pandas DataFrame, making it easy to manage and process the data. `df.head()` displays the first few rows of this DataFrame to give us a quick overview.

In [ ]:
augmented_queries = []

for q in df['query']:
    augmented_queries.append(q)
    augmented_queries.append("How to fix " + q.lower())
    augmented_queries.append("Issue: " + q.lower())
    augmented_queries.append(q + " problem")

df_aug = pd.DataFrame({
    "query": augmented_queries,
    "solution": df['solution'].repeat(4).values
})


This cell performs data augmentation on our `query` column. Data augmentation is a technique used to increase the amount of data by adding slightly modified copies of already existing data, or newly created synthetic data from existing data. This helps in making our model more robust and less prone to overfitting.

For each original query, we create three additional variations:
1.  The original query itself.
2.  'How to fix ' followed by the lowercase version of the query.
3.  'Issue: ' followed by the lowercase version of the query.
4.  The original query followed by ' problem'.

The `df_aug` DataFrame is then created with these augmented queries, and the corresponding `solution` is repeated to match the new queries. This expanded dataset provides more diverse phrasing for similar problems, which can improve the retrieval capabilities of our RAG system.

## 2. Data Preparation

We define a small dataset of common IT support queries and their corresponding solutions. This will serve as our knowledge base for the RAG system.

In [ ]:
df.info()
df.describe()


This cell performs initial exploratory data analysis (EDA) on the `df` DataFrame:

*   **`df.info()`**: Provides a concise summary of the DataFrame, including the number of entries, the number of columns, non-null values for each column, and the data types (`Dtype`). It also shows memory usage. This helps in quickly understanding the structure and completeness of the data.
*   **`df.describe()`**: Generates descriptive statistics of the DataFrame. For object (string) columns, it shows `count`, `unique` values, the `top` (most frequent) value, and its `freq` (frequency). This gives insights into the distribution and characteristics of the categorical data.

We perform basic data inspection to understand the structure and content of our `df` DataFrame.

In [ ]:
df.isnull().sum()


This cell checks for missing (null) values in each column of the DataFrame using `df.isnull().sum()`. The output shows that both 'query' and 'solution' columns have 0 null values, confirming the data quality for our small, custom dataset. This is an important step in any data preparation process to ensure that the data is clean and complete before further processing.

Checking for null values in the DataFrame to ensure data quality. Our small, custom dataset is expected to have no missing values.

In [ ]:
print(df.columns)


This line simply prints the column names of the DataFrame. In our case, it outputs `Index(['query', 'solution'], dtype='object')`, confirming that our DataFrame has the expected columns, 'query' and 'solution'.

Displaying the column names to verify they are as expected.

In [ ]:
df['query_length'] = df['query'].apply(len)

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6)) # Set a good figure size
sns.histplot(df['query_length'], kde=True, color='skyblue') # Use seaborn for better aesthetics
plt.title("Distribution of Query Lengths", fontsize=16) # More descriptive title
plt.xlabel("Query Length (Number of Characters)", fontsize=12) # Clearer x-axis label
plt.ylabel("Frequency", fontsize=12) # Clearer y-axis label
plt.grid(axis='y', linestyle='--', alpha=0.7) # Add a grid for readability
plt.tight_layout() # Adjust layout to prevent labels from being cut off
plt.show()

This code block performs an Exploratory Data Analysis (EDA) step to visualize the distribution of query lengths:

1.  **`df['query_length'] = df['query'].apply(len)`**: A new column named `query_length` is created in the DataFrame. For each query string in the `query` column, its length (number of characters) is calculated and stored in this new column.
2.  **`import matplotlib.pyplot as plt`** and **`import seaborn as sns`**: These lines import the necessary visualization libraries.
3.  **`plt.figure(figsize=(10, 6))`**: Sets the size of the plot for better readability.
4.  **`sns.histplot(df['query_length'], kde=True, color='skyblue')`**: Generates a histogram of the `query_length` column. A histogram shows the frequency distribution of a numerical variable. `kde=True` adds a Kernel Density Estimate (KDE) curve, which is a smoothed version of the histogram, estimating the probability density function of the variable.
5.  **`plt.title(...)`, `plt.xlabel(...)`, `plt.ylabel(...)`**: These lines set the title and axis labels for the plot, making it more informative.
6.  **`plt.grid(...)`** and **`plt.tight_layout()`**: Adds a grid for better readability and adjusts plot parameters for a tight layout.
7.  **`plt.show()`**: Displays the generated plot.

This visualization helps us understand the typical length of user queries, which can be useful for designing models or understanding user input patterns.

### Exploratory Data Analysis (EDA) for Queries

We analyze the length distribution of the queries to get an idea of their typical size. This can be useful for understanding the nature of the input data.

In [ ]:
from collections import Counter

words = " ".join(df['query']).split()
common_words = Counter(words).most_common(10)

print(common_words)


In [ ]:
words_df = pd.DataFrame(common_words, columns=['Word', 'Frequency'])

plt.figure(figsize=(12, 7))
sns.barplot(x='Frequency', y='Word', data=words_df, palette='viridis', hue='Word', legend=False)
plt.title('Top 10 Most Common Words in Queries', fontsize=16)
plt.xlabel('Frequency', fontsize=12)
plt.ylabel('Word', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

This code block continues the EDA by identifying and visualizing the most common words in the queries:

1.  **`from collections import Counter`**: Imports the `Counter` class, a convenient tool for counting hashable objects.
2.  **`words = " ".join(df['query']).split()`**: All queries are concatenated into a single string, and then this string is split into individual words. This creates a list of all words present in the queries.
3.  **`common_words = Counter(words).most_common(10)`**: `Counter(words)` creates a dictionary-like object where keys are words and values are their frequencies. `most_common(10)` then extracts the 10 words that appear most frequently.
4.  **`print(common_words)`**: Displays the list of the 10 most common words and their counts.
5.  **`words_df = pd.DataFrame(common_words, columns=['Word', 'Frequency'])`**: The `common_words` list is converted into a pandas DataFrame, making it suitable for plotting.
6.  **`plt.figure(...)`, `sns.barplot(...)`, `plt.title(...)`, `plt.xlabel(...)`, `plt.ylabel(...)`, `plt.grid(...)`, `plt.tight_layout()`**: These lines set up and generate a horizontal bar plot using `seaborn` to visualize the top 10 most common words and their frequencies. `hue='Word', legend=False` helps to color each bar uniquely without needing a separate legend.

This analysis helps us understand the prevalent vocabulary and themes within the IT support queries, giving insights into the types of issues users most commonly report.

Identifying the most common words in the queries helps us understand the prevalent themes and issues users are reporting.

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9 ]', '', text)
    return text

df['clean_query'] = df['query'].apply(clean_text)


This code block implements a text cleaning function and applies it to our queries:

1.  **`import re`**: Imports the regular expression module, which is powerful for string manipulation.
2.  **`def clean_text(text):`**: Defines a function `clean_text` that takes a string as input:
    *   **`text = text.lower()`**: Converts the entire text to lowercase. This standardizes the text, treating 'System' and 'system' as the same word, which is important for consistent embeddings.
    *   **`text = re.sub(r'[^a-zA-Z0-9 ]', '', text)`**: Uses a regular expression to remove any character that is NOT a letter (a-z, A-Z), a digit (0-9), or a space. This effectively removes punctuation, special symbols, etc.
    *   **`return text`**: Returns the cleaned text.
3.  **`df['clean_query'] = df['query'].apply(clean_text)`**: A new column `clean_query` is created in the DataFrame by applying the `clean_text` function to each query in the original `query` column.

Text cleaning is a crucial preprocessing step in NLP. It reduces noise in the data, standardizes text, and improves the quality of embeddings, leading to better performance in tasks like information retrieval and text generation.

### Text Cleaning

A text cleaning function is applied to the queries. This typically involves converting text to lowercase and removing special characters to standardize the input for embedding generation.

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = embed_model.encode(df['clean_query'].tolist())


This section is dedicated to generating text embeddings, a core component of our RAG system:

1.  **`from sentence_transformers import SentenceTransformer`**: Imports the `SentenceTransformer` class, which allows us to use pre-trained models to convert sentences into dense vector representations.
2.  **`embed_model = SentenceTransformer('all-MiniLM-L6-v2')`**: Loads a specific pre-trained Sentence Transformer model. 'all-MiniLM-L6-v2' is a lightweight yet powerful model that maps sentences and paragraphs to a 384-dimensional dense vector space. These vectors (embeddings) capture the semantic meaning of the text, meaning sentences with similar meanings will have similar vector representations.
3.  **`embeddings = embed_model.encode(df['clean_query'].tolist())`**: This is the key step where our cleaned queries are transformed into embeddings. The `encode` method takes a list of strings and returns a 2D NumPy array where each row is the embedding vector for a corresponding query.

These embeddings are vital because they allow us to perform mathematical operations (like calculating distances) to find queries that are semantically similar, even if their exact wording is different. This forms the retrieval part of RAG.

## 3. Embedding Generation

We use a pre-trained `SentenceTransformer` model (`all-MiniLM-L6-v2`) to convert our clean queries into numerical vector embeddings. These embeddings capture the semantic meaning of the text.

In [ ]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))


This code block sets up the FAISS index, which is critical for efficient similarity search:

1.  **`import faiss`**: Imports the FAISS library.
2.  **`import numpy as np`**: Imports NumPy for numerical operations, specifically for ensuring data types are correct for FAISS.
3.  **`dimension = embeddings.shape[1]`**: Retrieves the dimension of our embedding vectors. For 'all-MiniLM-L6-v2', this will be 384.
4.  **`index = faiss.IndexFlatL2(dimension)`**: Initializes a FAISS index. `IndexFlatL2` is one of the simplest FAISS index types, performing an exhaustive search using L2 (Euclidean) distance. It's suitable for smaller datasets or when high precision is required, as it compares the query vector to every vector in the index.
5.  **`index.add(np.array(embeddings))`**: Adds our `embeddings` (which are NumPy arrays) to the FAISS index. Once added, the index is ready to perform similarity searches against these stored vectors.

FAISS is designed for fast nearest neighbor search in large datasets of vectors, making it highly efficient for retrieving the most relevant context from our knowledge base in real-time.

## 4. FAISS Index for Efficient Retrieval

FAISS (Facebook AI Similarity Search) is used to create an index for our embeddings. This allows for fast and efficient similarity search, which is crucial for retrieving relevant solutions quickly.

In [ ]:
def retrieve(query, k=2):
    query_vec = embed_model.encode([query])
    distances, indices = index.search(np.array(query_vec), k)

    results = df.iloc[indices[0]]
    return results


This cell defines the `retrieve` function, which is responsible for finding the most relevant solutions for a given user query:

1.  **`def retrieve(query, k=2):`**: Defines the function, taking a `query` string and an optional `k` parameter (defaulting to 2) which specifies how many top similar items to retrieve.
2.  **`query_vec = embed_model.encode([query])`**: The incoming user `query` is first encoded into an embedding vector using the same `SentenceTransformer` model that was used to create the index. This ensures consistency in the vector space.
3.  **`distances, indices = index.search(np.array(query_vec), k)`**: This is where FAISS does its work. `index.search()` takes the query vector(s) and `k` (the number of nearest neighbors to find). It returns two arrays:
    *   `distances`: The L2 distances between the query vector and the `k` nearest neighbors.
    *   `indices`: The indices (positions) of the `k` nearest neighbors in the original `embeddings` array.
4.  **`results = df.iloc[indices[0]]`**: Uses the retrieved `indices` to select the corresponding rows from our original `df` DataFrame. `indices[0]` is used because `index.search` returns a 2D array of indices (even for a single query).
5.  **`return results`**: The function returns a DataFrame containing the `query` and `solution` of the `k` most similar entries from our knowledge base.

This function effectively implements the 'Retrieval' part of the RAG system, fetching relevant information that will then be used to inform the text generation model.

### Retrieval Function

The `retrieve` function takes a user query, converts it into an embedding, and then uses the FAISS index to find the `k` most similar queries (and their corresponding solutions) from our dataset.

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")


This code block loads our text generation model, which will be responsible for formulating the final response:

1.  **`from transformers import pipeline`**: Imports the `pipeline` utility from the `transformers` library. The `pipeline` is an easy-to-use API for performing common NLP tasks, abstracting away the complexities of model loading and preprocessing.
2.  **`generator = pipeline("text-generation", model="gpt2")`**: Initializes a text generation pipeline using the `gpt2` model. `gpt2` is a well-known large language model from OpenAI, capable of generating coherent and contextually relevant text. The `text-generation` pipeline wraps the model and its tokenizer, allowing us to simply provide a prompt and get a generated text response.

This `generator` will take the retrieved context and the user's original query to generate a human-like and informative answer, completing the 'Generation' part of the RAG framework.

## 5. Text Generation Model

We load a pre-trained `gpt2` model using `transformers.pipeline` for text generation. This model will be used to formulate a helpful response based on the retrieved context.

In [ ]:
def rag_chatbot(query):
    retrieved = retrieve(query)

    context = "\n".join(retrieved['solution'].tolist())

    prompt = f"""
    You are an IT support assistant.

    Context:
    {context}

    User Query:
    {query}

    Provide a helpful solution:
    """

    response = generator(prompt, max_length=150)[0]['generated_text']

    return response


This is the core `rag_chatbot` function, bringing together the retrieval and generation components:

1.  **`def rag_chatbot(query):`**: Defines the chatbot function, taking a `query` string from the user.
2.  **`retrieved = retrieve(query)`**: Calls our previously defined `retrieve` function to get the most relevant entries from our knowledge base based on the user's `query`. This returns a DataFrame of similar queries and their solutions.
3.  **`context = "\n".join(retrieved['solution'].tolist())`**: Extracts the 'solution' column from the `retrieved` DataFrame, converts it to a list, and then joins these solutions into a single string, separated by newlines. This combined string forms the 'context' for our generation model.
4.  **`prompt = f"""..."""`**: Constructs a `prompt` string that will be fed to the `gpt2` generator. The prompt is carefully designed to instruct the `gpt2` model on its role (IT support assistant) and to provide it with the necessary context and the user's query. This structure helps the `gpt2` model generate a relevant and helpful response.
    *   **`You are an IT support assistant.`**: Sets the persona for the AI.
    *   **`Context:
    {context}`**: Injects the retrieved solutions directly into the prompt.
    *   **`User Query:
    {query}`**: Includes the original user query.
    *   **`Provide a helpful solution:`**: Instructs the model on what kind of output is expected.
5.  **`response = generator(prompt, max_length=150)[0]['generated_text']`**: Calls the `generator` pipeline (our `gpt2` model) with the constructed `prompt`. `max_length=150` limits the length of the generated response to 150 tokens. The `pipeline` returns a list of dictionaries, so `[0]['generated_text']` extracts the actual generated text.
6.  **`return response`**: Returns the AI's generated solution.

This function embodies the RAG architecture: it first *retrieves* relevant information and then *generates* a response conditioned on that retrieved information.

## 6. RAG Chatbot Implementation

The `rag_chatbot` function integrates the retrieval and generation components. It first retrieves relevant solutions, constructs a prompt with this context and the user's query, and then uses the `gpt2` model to generate a response.

In [ ]:
print(rag_chatbot("My laptop is very slow"))


This cell serves as a quick test for our `rag_chatbot` function. It calls the chatbot with the query "My laptop is very slow" and prints the generated response.

This allows us to immediately see the chatbot in action and verify if it's producing a coherent and somewhat relevant answer based on the context it retrieves. It's a useful sanity check before moving on to more formal evaluation.

### Test the Chatbot

Here, we test the `rag_chatbot` with an example query to see its generated response.

In [ ]:
correct = 0

for i in range(len(df)):
    pred = rag_chatbot(df['query'][i])

    if df['solution'][i].lower() in pred.lower():
        correct += 1

accuracy = correct / len(df)
print("Accuracy:", accuracy)


This code block evaluates the basic accuracy of our `rag_chatbot` against the original dataset. It's a simple, rule-based evaluation to check if the generated response contains the expected solution.

1.  **`correct = 0`**: Initializes a counter for correct predictions.
2.  **`for i in range(len(df)):`**: Loops through each query and its corresponding solution in our original `df` DataFrame.
3.  **`pred = rag_chatbot(df['query'][i])`**: For each original query, it calls the `rag_chatbot` to get a generated `pred`iction (response).
4.  **`if df['solution'][i].lower() in pred.lower():`**: This is the evaluation logic. It checks if the *actual solution* (from our DataFrame) is present anywhere within the *generated response*. Both are converted to lowercase for case-insensitive comparison. If the actual solution is found, `correct` is incremented.
5.  **`accuracy = correct / len(df)`**: After iterating through all queries, the `accuracy` is calculated as the ratio of `correct` predictions to the total number of queries.
6.  **`print("Accuracy:", accuracy)`**: Prints the calculated accuracy.

This metric provides a preliminary understanding of how well the chatbot is able to recall and include the correct solutions from its knowledge base in its generated answers. A 1.0 accuracy indicates that for every query in the dataset, the expected solution was present in the chatbot's response.

## 7. Evaluate Chatbot Performance

We evaluate the chatbot's ability to provide correct solutions by iterating through our dataset and checking if the actual solution is present in the generated response. This provides a basic accuracy metric for our RAG system.